# 05 · Cross-subject model evaluation

Three outer folds cover all 51 WISDM and 10 MHEALTH participants. For each dataset and fold, three training seeds give 18 quantized networks. The fixed validation and test partitions are disjoint from training. The network architecture and data preparation are documented in the repository.

This notebook checks every saved split, model array, and held-out fixture. Set `TINYML_NOTEBOOK_REBUILD=1` to retrain all 18 models from the original UCI archives and compare the exports.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "study.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tinyml.config import DATASETS, SEEDS
from tinyml.folds import CLIP_DIR, FOLDS, model_dir, read_fixture
from tinyml.study_v2 import V2

REBUILD = os.getenv("TINYML_NOTEBOOK_REBUILD") == "1"
WORKERS = os.getenv("TINYML_NOTEBOOK_WORKERS", "3")
DATA = Path(os.getenv("TINYML_DATA_DIR", ROOT / "data")).resolve()
SCRATCH = ROOT / "build" / "notebooks_v2"

def run(*args):
    proc = subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout + proc.stderr, end="")
    proc.check_returncode()

print("Saved-result verification" if not REBUILD else "Rebuild and verification")

Rebuild and verification


In [2]:
from tinyml.quant import Int8Model, dense_inference
from tinyml.protocol import sensor_input

rows = []
for dataset in DATASETS:
    covered = set()
    for fold in FOLDS:
        clips = read_fixture(dataset, fold)
        for seed in SEEDS:
            path = model_dir(V2, dataset, fold, seed)
            row = json.loads((path / "metrics.json").read_text())
            split = {name: set(ids) for name, ids in row["subjects"].items()}
            assert not split["train"] & split["validation"]
            assert not split["train"] & split["test"]
            assert not split["validation"] & split["test"]
            assert set(clip.subject for clip in clips) == split["test"]
            if seed == SEEDS[0]:
                assert not covered & split["test"]
                covered |= split["test"]
            model = Int8Model.load(path / "int8.npz")
            with np.load(path / "fp32.npz") as weights:
                assert len(weights.files) == 8
            sample = sensor_input(model, clips[0])
            assert dense_inference(model, sample).shape == (120, len(row["labels"]))
            rows.append({
                "dataset": dataset, "fold": fold, "seed": seed,
                "test subjects": len(split["test"]), "clips": len(clips),
                "INT8 accuracy (%)": 100 * row["int8"]["accuracy"],
                "INT8 macro F1 (%)": 100 * row["int8"]["macro_f1"],
            })
    print(dataset, "distinct test subjects:", len(covered))
display(pd.DataFrame(rows).round(2))

wisdm distinct test subjects: 51
mhealth distinct test subjects: 10


,dataset,fold,seed,test subjects,clips,INT8 accuracy (%),INT8 macro F1 (%)
0,wisdm,0,13,17,54,46.37,45.39
1,wisdm,0,37,17,54,47.11,46.89
2,wisdm,0,73,17,54,46.83,45.67
3,wisdm,1,13,17,54,43.85,42.88
4,wisdm,1,37,17,54,45.83,45.31
5,wisdm,1,73,17,54,45.66,44.89
6,wisdm,2,13,17,54,44.94,44.54
7,wisdm,2,37,17,54,46.68,45.68
8,wisdm,2,73,17,54,46.71,45.85
9,mhealth,0,13,4,36,41.96,42.41


In [3]:
if REBUILD:
    output = SCRATCH / "training"
    run("-m", "tinyml.train_folds", "--data-dir", DATA, "--results", output, "--workers", WORKERS)
    for dataset in DATASETS:
        for fold in FOLDS:
            for seed in SEEDS:
                reference = model_dir(V2, dataset, fold, seed)
                trial = model_dir(output, dataset, fold, seed)
                for name in ("int8.npz", "fp32.npz"):
                    with np.load(reference / name) as expected, np.load(trial / name) as observed:
                        for key in expected.files:
                            np.testing.assert_array_equal(expected[key], observed[key])
    print("All 18 training runs reproduced the archived tensors on this host.")

wisdm fold=0 seed=73 epoch=1: validation F1=0.211
wisdm fold=1 seed=13 epoch=1: validation F1=0.212
wisdm fold=0 seed=37 epoch=1: validation F1=0.216
wisdm fold=1 seed=37 epoch=1: validation F1=0.219
wisdm fold=0 seed=13 epoch=1: validation F1=0.214
wisdm fold=0 seed=73 epoch=10: validation F1=0.370
wisdm fold=1 seed=13 epoch=10: validation F1=0.392
wisdm fold=0 seed=37 epoch=10: validation F1=0.379
wisdm fold=1 seed=37 epoch=10: validation F1=0.394
wisdm fold=0 seed=73 epoch=20: validation F1=0.392
wisdm fold=1 seed=13 epoch=20: validation F1=0.419
wisdm fold=0 seed=13 epoch=10: validation F1=0.359
wisdm fold=0 seed=73 epoch=26: validation F1=0.396
wisdm fold=1 seed=13 epoch=26: validation F1=0.423
Finished wisdm/0/73 held-out INT8 accuracy 0.468
Finished wisdm/1/13 held-out INT8 accuracy 0.438
wisdm fold=1 seed=73 epoch=1: validation F1=0.177
wisdm fold=2 seed=13 epoch=1: validation F1=0.259
wisdm fold=0 seed=37 epoch=20: validation F1=0.408
wisdm fold=1 seed=37 epoch=20: validation 